In [3]:

import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from safetensors.torch import load_file
from transformers import BertForSequenceClassification, BertTokenizer


# ============================================================
# 1. 설정
# ============================================================

SEED = 42
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
NUM_LABELS = 6

PREDICTIONS_PATH = Path(
    "results/error_analysis/validation_predictions.csv"
)

CHECKPOINT_PATH = Path(
    "results/augmentation/wordnet_comparison/baseline/"
    "best_model.safetensors"
)

OUTPUT_DIR = Path("results/attention_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {DEVICE}")
print(f"Predictions: {PREDICTIONS_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output: {OUTPUT_DIR}")


# ============================================================
# 2. 입력 파일 확인 및 데이터 로드
# ============================================================

if not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        f"예측 파일을 찾을 수 없습니다: {PREDICTIONS_PATH}"
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"체크포인트를 찾을 수 없습니다: {CHECKPOINT_PATH}"
    )

df = pd.read_csv(PREDICTIONS_PATH)

print("\n[입력 CSV 컬럼]")
print(df.columns.tolist())


def find_column(candidates, columns, description):
    """후보 컬럼명 중 실제 데이터에 존재하는 컬럼을 찾는다."""
    for name in candidates:
        if name in columns:
            return name

    raise ValueError(
        f"{description} 컬럼을 찾을 수 없습니다.\n"
        f"후보 컬럼: {candidates}\n"
        f"실제 컬럼: {columns.tolist()}"
    )


text_col = find_column(
    ["text", "sentence", "content"],
    df.columns,
    "텍스트",
)

true_col = find_column(
    ["true_label", "true", "label", "actual_label"],
    df.columns,
    "실제 라벨",
)

pred_col = find_column(
    ["predicted_label", "pred_label", "prediction", "pred"],
    df.columns,
    "예측 라벨",
)

df = df.dropna(
    subset=[text_col, true_col, pred_col]
).copy()

df[true_col] = pd.to_numeric(
    df[true_col], errors="raise"
).astype(int)

df[pred_col] = pd.to_numeric(
    df[pred_col], errors="raise"
).astype(int)

df[text_col] = df[text_col].astype(str)

if not df[true_col].between(0, NUM_LABELS - 1).all():
    raise ValueError(
        f"실제 라벨은 0~{NUM_LABELS - 1} 범위여야 합니다."
    )

if not df[pred_col].between(0, NUM_LABELS - 1).all():
    raise ValueError(
        f"예측 라벨은 0~{NUM_LABELS - 1} 범위여야 합니다."
    )

print(f"\n검증 데이터: {len(df):,}개")
print(f"텍스트 컬럼: {text_col}")
print(f"실제 라벨 컬럼: {true_col}")
print(f"기존 예측 컬럼: {pred_col}")


# ============================================================
# 3. 분석 사례 선정
# ============================================================

# 주요 오분류 유형에서 최대 2개씩 추출한다.
CONFUSION_PAIRS = [
    (1, 2),
    (4, 5),
    (0, 3),
]

selected = []

for true_label, pred_label in CONFUSION_PAIRS:
    pair_df = df[
        (df[true_col] == true_label)
        & (df[pred_col] == pred_label)
    ]

    if pair_df.empty:
        print(
            f"[주의] {true_label}->{pred_label} "
            "오분류 사례가 없습니다."
        )
        continue

    samples = pair_df.sample(
        n=min(2, len(pair_df)),
        random_state=SEED,
    )

    for _, row in samples.iterrows():
        selected.append({
            "text": row[text_col],
            "true_label": int(row[true_col]),
            "saved_predicted_label": int(row[pred_col]),
            "case_type": "misclassified",
        })


# 각 클래스에서 정분류 사례를 최대 1개씩 추가한다.
for label in range(NUM_LABELS):
    correct_df = df[
        (df[true_col] == label)
        & (df[pred_col] == label)
    ]

    if correct_df.empty:
        print(
            f"[주의] 클래스 {label}의 정분류 사례가 없습니다."
        )
        continue

    sample = correct_df.sample(
        n=1,
        random_state=SEED + label,
    ).iloc[0]

    selected.append({
        "text": sample[text_col],
        "true_label": int(sample[true_col]),
        "saved_predicted_label": int(sample[pred_col]),
        "case_type": "correct",
    })


if not selected:
    raise ValueError("분석할 사례가 없습니다.")

cases = pd.DataFrame(selected)
cases.insert(0, "case_id", range(1, len(cases) + 1))

cases_path = OUTPUT_DIR / "selected_cases.csv"

cases.to_csv(
    cases_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"\n선정된 분석 사례: {len(cases)}개")
print(
    cases[
        [
            "case_id",
            "true_label",
            "saved_predicted_label",
            "case_type",
        ]
    ].to_string(index=False)
)


# ============================================================
# 4. BERT 모델 및 체크포인트 로드
# ============================================================

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# 중요:
# SDPA 구현 대신 eager 구현을 사용하여
# output_attentions=True에서 Attention 가중치를 반환받는다.
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    attn_implementation="eager",
)

state_dict = load_file(str(CHECKPOINT_PATH))

# strict=True:
# 분류기 파라미터를 포함하여 모델 구조가 일치해야 한다.
model.load_state_dict(
    state_dict,
    strict=True,
)

model.to(DEVICE)
model.eval()

print("\n기존 학습 체크포인트 로드 완료")
print(f"모델 클래스: {model.__class__.__name__}")
print(f"분류 클래스 수: {model.config.num_labels}")
print(
    "Attention 구현:",
    model.config._attn_implementation,
)

assert model.config.num_labels == NUM_LABELS
assert model.config._attn_implementation == "eager"


# ============================================================
# 5. Attention 분석
# ============================================================

all_token_scores = []
case_summaries = []

special_tokens = set(tokenizer.all_special_tokens)

for _, case in cases.iterrows():

    case_id = int(case["case_id"])
    text = case["text"]

    print(
        f"\n[{case_id}/{len(cases)}] "
        "Attention 분석 중..."
    )

    # --------------------------------------------------------
    # 5-1. 토큰화
    # --------------------------------------------------------

    encoded = tokenizer(
        text,
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    encoded = {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }

    # --------------------------------------------------------
    # 5-2. 예측 및 Attention 추출
    # --------------------------------------------------------

    with torch.inference_mode():
        outputs = model(
            **encoded,
            output_attentions=True,
            return_dict=True,
        )

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1,
    )[0]

    model_predicted_label = int(
        probabilities.argmax().item()
    )

    model_confidence = float(
        probabilities[model_predicted_label].item()
    )

    if (
        outputs.attentions is None
        or len(outputs.attentions) == 0
    ):
        raise RuntimeError(
            "Attention 가중치가 반환되지 않았습니다. "
            "attn_implementation='eager' 설정을 확인하세요."
        )

    # 마지막 Transformer 층:
    # [batch_size, num_heads, seq_len, seq_len]
    last_layer_attention = outputs.attentions[-1][0]

    # 모든 Attention head의 가중치를 평균한다.
    # [seq_len, seq_len]
    mean_attention = last_layer_attention.mean(dim=0)

    # [CLS]가 각 토큰 위치에 부여한 Attention.
    # [seq_len]
    cls_attention = mean_attention[0]

    # --------------------------------------------------------
    # 5-3. 토큰별 점수 추출
    # --------------------------------------------------------

    input_ids = encoded["input_ids"][0].cpu().tolist()
    attention_mask = (
        encoded["attention_mask"][0].cpu().tolist()
    )

    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    valid_positions = [
        i
        for i, mask in enumerate(attention_mask)
        if mask == 1
    ]

    token_rows = []

    for position in valid_positions:
        token = tokens[position]

        # 특수 토큰은 결과에서 제외한다.
        if token in special_tokens:
            continue

        score = float(
            cls_attention[position].item()
        )

        token_rows.append({
            "case_id": case_id,
            "text": text,
            "true_label": int(case["true_label"]),
            "saved_predicted_label": int(
                case["saved_predicted_label"]
            ),
            "model_predicted_label": model_predicted_label,
            "model_confidence": model_confidence,
            "case_type": case["case_type"],
            "token_position": position,
            "token": token,
            "attention_score": score,
        })

    all_token_scores.extend(token_rows)

    # --------------------------------------------------------
    # 5-4. 사례별 요약
    # --------------------------------------------------------

    saved_predicted_label = int(
        case["saved_predicted_label"]
    )

    prediction_matches = (
        saved_predicted_label == model_predicted_label
    )

    case_summaries.append({
        "case_id": case_id,
        "text": text,
        "true_label": int(case["true_label"]),
        "saved_predicted_label": saved_predicted_label,
        "model_predicted_label": model_predicted_label,
        "model_confidence": model_confidence,
        "prediction_matches_saved": prediction_matches,
        "case_type": case["case_type"],
        "token_count": len(token_rows),
    })

    print(f"  실제 라벨: {int(case['true_label'])}")
    print(f"  기존 예측: {saved_predicted_label}")
    print(f"  현재 예측: {model_predicted_label}")
    print(f"  예측 일치: {prediction_matches}")
    print(f"  예측 확률: {model_confidence:.4f}")
    print(f"  분석 토큰 수: {len(token_rows)}")

    # --------------------------------------------------------
    # 5-5. 사례별 Attention 그래프
    # --------------------------------------------------------

    if token_rows:
        token_df = pd.DataFrame(token_rows)

        top_tokens = token_df.nlargest(
            min(15, len(token_df)),
            "attention_score",
        ).sort_values("attention_score")

        fig, ax = plt.subplots(figsize=(9, 6))

        ax.barh(
            top_tokens["token"],
            top_tokens["attention_score"],
        )

        ax.set_title(
            f"Case {case_id} | "
            f"True={case['true_label']} | "
            f"Saved={saved_predicted_label}\n"
            f"Current={model_predicted_label} | "
            f"Confidence={model_confidence:.3f}"
        )

        ax.set_xlabel(
            "Last-layer mean attention from [CLS]"
        )
        ax.set_ylabel("Token")

        fig.tight_layout()

        fig.savefig(
            OUTPUT_DIR / f"case_{case_id:03d}_attention.png",
            dpi=150,
            bbox_inches="tight",
        )

        plt.close(fig)


# ============================================================
# 6. 결과 저장
# ============================================================

token_scores_df = pd.DataFrame(all_token_scores)
summary_df = pd.DataFrame(case_summaries)

token_scores_df.to_csv(
    OUTPUT_DIR / "token_attention_scores.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_df.to_csv(
    OUTPUT_DIR / "attention_case_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

if not token_scores_df.empty:
    top_tokens_df = (
        token_scores_df
        .sort_values(
            ["case_id", "attention_score"],
            ascending=[True, False],
        )
        .groupby("case_id")
        .head(10)
    )

    top_tokens_df.to_csv(
        OUTPUT_DIR / "top_attention_tokens.csv",
        index=False,
        encoding="utf-8-sig",
    )
else:
    print("[주의] 저장할 토큰 Attention 점수가 없습니다.")


# ============================================================
# 7. 결과 검증 및 요약
# ============================================================

print("\n" + "=" * 60)
print("Attention 분석 완료")
print("=" * 60)

print(f"결과 저장 위치: {OUTPUT_DIR.resolve()}")
print(f"분석 사례 수: {len(cases)}개")
print(f"Attention 토큰 수: {len(token_scores_df):,}개")

if not summary_df.empty:
    mismatch_count = int(
        (~summary_df["prediction_matches_saved"]).sum()
    )

    print(
        "기존 예측과 현재 예측이 다른 사례:",
        f"{mismatch_count}/{len(summary_df)}",
    )

    if mismatch_count > 0:
        print(
            "[주의] 예측 결과가 다른 사례가 있습니다. "
            "모델 체크포인트, 토큰화 및 전처리 설정을 "
            "확인한 후 Attention 결과를 해석하세요."
        )
    else:
        print("선정된 모든 사례의 예측이 기존 결과와 일치합니다.")

print("\n[생성 파일]")

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"- {path.name}")


Device: cuda
Predictions: results/error_analysis/validation_predictions.csv
Checkpoint: results/augmentation/wordnet_comparison/baseline/best_model.safetensors
Output: results/attention_analysis

[입력 CSV 컬럼]
['text', 'true_label', 'true_class', 'predicted_label', 'predicted_class', 'correct', 'confidence', 'top2_margin', 'prediction_entropy', 'prob_label_0', 'prob_label_1', 'prob_label_2', 'prob_label_3', 'prob_label_4', 'prob_label_5']

검증 데이터: 65,422개
텍스트 컬럼: text
실제 라벨 컬럼: true_label
기존 예측 컬럼: predicted_label

선정된 분석 사례: 12개
 case_id  true_label  saved_predicted_label     case_type
       1           1                      2 misclassified
       2           1                      2 misclassified
       3           4                      5 misclassified
       4           4                      5 misclassified
       5           0                      3 misclassified
       6           0                      3 misclassified
       7           0                      0       correct
  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



기존 학습 체크포인트 로드 완료
모델 클래스: BertForSequenceClassification
분류 클래스 수: 6
Attention 구현: eager

[1/12] Attention 분석 중...
  실제 라벨: 1
  기존 예측: 2
  현재 예측: 2
  예측 일치: True
  예측 확률: 0.6146
  분석 토큰 수: 34

[2/12] Attention 분석 중...
  실제 라벨: 1
  기존 예측: 2
  현재 예측: 2
  예측 일치: True
  예측 확률: 0.6185
  분석 토큰 수: 35

[3/12] Attention 분석 중...
  실제 라벨: 4
  기존 예측: 5
  현재 예측: 5
  예측 일치: True
  예측 확률: 0.5798
  분석 토큰 수: 15

[4/12] Attention 분석 중...
  실제 라벨: 4
  기존 예측: 5
  현재 예측: 5
  예측 일치: True
  예측 확률: 0.5824
  분석 토큰 수: 19

[5/12] Attention 분석 중...
  실제 라벨: 0
  기존 예측: 3
  현재 예측: 3
  예측 일치: True
  예측 확률: 0.8689
  분석 토큰 수: 20

[6/12] Attention 분석 중...
  실제 라벨: 0
  기존 예측: 3
  현재 예측: 3
  예측 일치: True
  예측 확률: 0.5630
  분석 토큰 수: 39

[7/12] Attention 분석 중...
  실제 라벨: 0
  기존 예측: 0
  현재 예측: 0
  예측 일치: True
  예측 확률: 0.9675
  분석 토큰 수: 26

[8/12] Attention 분석 중...
  실제 라벨: 1
  기존 예측: 1
  현재 예측: 1
  예측 일치: True
  예측 확률: 0.9828
  분석 토큰 수: 22

[9/12] Attention 분석 중...
  실제 라벨: 2
  기존 예측: 2
  현재 예측: 2
  예측 일치: True
  예측 확률: 0.985